In [1]:
import pandas as pd
import sqlite3

/root/.venv/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


### Создадим подключение к БД с помощью sqlite3

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

### Получим схему таблицы checker

In [3]:
pd.read_sql('PRAGMA table_info(checker)', conn)

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,status,TEXT,0,None,0
2,2,success,INTEGER,0,None,0
3,3,timestamp,TIMESTAMP,0,None,0
4,4,numTrials,INTEGER,0,None,0
5,5,labname,TEXT,0,None,0
6,6,uid,TEXT,0,None,0


### Получим первые 10 строк из таблицы checker

In [4]:
pd.read_sql('SELECT * FROM checker LIMIT 10', conn)

,index,status,success,timestamp,numTrials,labname,uid
0,0,checking,0,2020-04-16 21:12:50.740474,5,None,admin_1
1,1,ready,0,2020-04-16 21:12:54.708365,5,code_rvw,admin_1
2,2,checking,0,2020-04-16 21:46:47.769088,7,None,admin_1
3,3,ready,0,2020-04-16 21:46:48.121217,7,lab02,admin_1
4,4,checking,0,2020-04-16 21:53:01.862637,6,code_rvw,admin_1
5,5,ready,0,2020-04-16 21:53:05.373389,6,code_rvw,admin_1
6,6,checking,0,2020-04-17 05:18:51.965864,1,None,None
7,7,ready,0,2020-04-17 05:19:02.744528,1,project1,user_4
8,8,checking,0,2020-04-17 05:22:35.249331,2,project1,user_4
9,9,ready,1,2020-04-17 05:22:45.549397,2,project1,user_4


### Подсчитаем сколько строк удовлетворяет требованиям, используя запрос с подзапросами

* status = «ready», мы не хотим анализировать журналы, которые находятся в процессе проверки статуса
* numTrials = 1, мы хотим проанализировать только первые коммиты, потому что только они могут сказать нам, когда студент начал работать над лабораторной работой
* названия лабораторных работ должны быть из списка: «laba04», «laba04s», «laba05», «laba06», «laba06s», «project1». Только они были активны во время эксперимента

In [5]:
query = """
SELECT COUNT(uid) as cnt
FROM pageviews
WHERE uid IN
    (SELECT uid
    FROM checker
    WHERE status = 'ready' AND numTrials = 1 AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1'))
ORDER BY uid ASC
"""
pd.io.sql.read_sql(query, conn, index_col="cnt")

""
cnt
985


### Закроем соединение с базой данных

In [6]:
conn.close()